In [ ]:
#Import Libraries here

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
import os
import sys
pio.renderers.default = "notebook_connected"

In [ ]:
current_dir = os.path.abspath('')
project_root = os.path.abspath(os.path.join(current_dir, '../../'))

if project_root not in sys.path:
    sys.path.append(project_root)

os.chdir(project_root)

print(f"Working directory set to: {os.getcwd()}")

In [ ]:
#Import project modules here

import importlib
from src.portfolio_simulation_utils import portfolio_simulation as p_sim
from src.data_pipeline_utils import data_fetching_handling as data_pipe
importlib.reload(p_sim)
importlib.reload(data_pipe)

# Project Overview

This project explores how mathematical concepts such as linear algebra, probability, and calculus are applied in quantitative finance.

The analysis focuses on three financial domains:

- Portfolio optimization (Markowitz Modern Portfolio Theory)
- Option pricing (Black–Scholes model)
- Fixed income instruments with embedded options

Across these topics, the same mathematical tools appear repeatedly: matrix algebra, probability distributions, and derivatives of functions. The project therefore studies how these tools allow us to model financial risk and portfolio behavior.

# Project Overview

This project explores how mathematical concepts such as linear algebra, probability, and calculus are applied in quantitative finance.

The analysis focuses on three financial domains:

- Portfolio optimization (Markowitz Modern Portfolio Theory)
- Option pricing (Black–Scholes model)
- Fixed income instruments with embedded options

Across these topics, the same mathematical tools appear repeatedly: matrix algebra, probability distributions, and derivatives of functions. The project therefore studies how these tools allow us to model financial risk and portfolio behavior.

# Central Question

How can mathematical tools such as matrix algebra, probability theory, and calculus be used to model risk and optimize financial portfolios?

# Thesis

Financial risk is fundamentally nonlinear. While linear approximations such as expected return or duration provide local insight, the true behavior of financial systems is driven by second-order structure such as variance, covariance, convexity, and option gamma.

This project demonstrates how these nonlinear effects emerge in three settings:

1. Portfolio optimization
2. Option pricing
3. Fixed income instruments with embedded options

# Methodology

The project follows three analytical approaches:

1. Empirical simulation  
   Monte Carlo simulation is used to explore the feasible set of portfolio risk–return combinations.

2. Analytical derivation  
   Closed-form mathematical models such as the Markowitz efficient frontier and the Black–Scholes option pricing formula are examined.

3. Numerical optimization  
   Constrained optimization methods are used to compute optimal portfolios subject to realistic investment constraints.

The main mathematical tools used include:

- Linear algebra (matrix notation and covariance matrices)
- Probability theory (normal distributions and cumulative distribution functions)
- Calculus (first and second derivatives)
- Taylor approximations

# Project Structure

The project is organized into several notebooks:

**1_1 – Efficient Frontier (Practical Approach)**  
Monte Carlo simulation of random portfolios to visualize the feasible region.

**1_2 – Efficient Frontier (Mathematical Approach)**  
Derivation of the efficient frontier using the Markowitz closed-form solution.

**1_3 – Efficient Frontier (Algorithmic Approach)**  
Numerical optimization of portfolio weights under constraints.

**2_1 – Black–Scholes and Portfolio Protection**  
Application of option pricing to hedge portfolio downside risk.

**2_2 – Option Pricing in Bonds**  
Application of option theory to callable bonds and convexity effects.

### 1. Let us first choose our stocks and extract the necessary data utilizing the yfinance librabry in Python ###

In the list ticker we can implement the symbol tickers of stocks we like. Since this is more of a scientific experiment and not a data science project, #let us stick to stocks from the US equity market only# in order not to deal with mismatching working days.  

For each ticker we calculate the daily returns utilizing our util functions in the data pipeline module. Afterwards we have to make sure that the result of the print is 1. This means that each ticker has price history for the full selected period, which by default is going to be 10 years.

If the result is larger than 1, then one or more tickers does not have a full period trading data and you must inspect the values in the all_df_shapes dictionary and replace the tickers with shorter time periods or rerun the same list of tickers by selecting a shorter period of time, which is less preferable for this theoretical experiment. 

In [ ]:
tickers = ['AAPL', 'NVDA', 'MSFT', 'JNJ', 'BAC', 'VZ', 'WMT', 'UPS', 'PFE', 'JPM']
all_df_shapes = {}

for ticker in tickers:
    data = data_pipe.save_10_year_single_stock_data_to_csv(ticker)
    return_data = data_pipe.create_returns_and_save(data, ticker)
    all_df_shapes[ticker] = return_data.shape

all_same = set(all_df_shapes.values())
print(len(all_same))

### Let us randomly check if we have what we need ###

In [ ]:
MSFT_data = data_pipe.fetch_raw_data('MSFT')
return_data_PFE = data_pipe.fetch_returns_data('PFE')

print(return_data_PFE)

In [ ]:
def create_candlestick_graph(ticker):
    data = data_pipe.fetch_raw_data(ticker)

    
    fig = go.Figure(
        data=[
            go.Candlestick(
                x=data.index,
                open=data["Open"],
                high=data["High"],
                low=data["Low"],
                close=data["Close"],
                name=f"{ticker}"
            )
        ]
    )
    
    fig.update_layout(
        title=F"{ticker} Daily Candlestick",
        xaxis_title="Date",
        yaxis_title="Price",
        xaxis_rangeslider_visible=False
    )
    
    fig.show()

def create_histogram_distribution_daily_log_returns(ticker):
    data = data_pipe.fetch_returns_data(ticker)
    
    plt.figure(figsize=(10,6))
    plt.hist(data["log_return_pct"], bins=100)
    plt.title(F"Distribution of {ticker} Daily Log Returns")
    plt.xlabel("Log Return")
    plt.ylabel("Frequency")
    plt.show()


def create_correlation_heatmap(corr_matrix):
    plt.figure(figsize=(8,6))
    sns.heatmap(
        corr_matrix,
        annot=True,
        cmap="coolwarm",
        vmin=-1,
        vmax=1,
        linewidths=0.5
    )

    plt.title("Correlation Matrix Heatmap")
    plt.show()

create_candlestick_graph('MSFT')

In [ ]:
MSFT_return_data = data_pipe.fetch_returns_data('MSFT')
mean = MSFT_return_data["log_return"].mean()
variance = MSFT_return_data["log_return"].var()
st_dev = MSFT_return_data["log_return"].std()
print(f"Mean log return in observation period: {mean},\n"
      f"Variance of log returns in observation period: {variance},\n"
      f"Standard deviation of log returns in observation period: {st_dev}")


create_histogram_distribution_daily_log_returns('MSFT')

In [ ]:
MSFT_return_data["log_return"].describe()

### Covariance Matrix of Stock Returns ###

$$ X =
\begin{bmatrix}
r_{1,1} & r_{2,1} \\
r_{1,2} & r_{2,2} \\
\vdots  & \vdots  \\
r_{1,n} & r_{2,n}
\end{bmatrix} $$


$$ \Sigma = \frac{1}{n-1}(X - \bar{X})^\top (X - \bar{X}) $$

In [ ]:
returns_df =  data_pipe.build_returns_df(tickers)

print(returns_df)

cov_matrix = returns_df.cov()
print(cov_matrix)

eigenvalues = np.linalg.eigvals(cov_matrix)
print(eigenvalues)

### Correlation (Normalized Covariance) ###

$$ \rho_{ij} = \frac{\mathrm{Cov}_{ij}}{\sigma_i \sigma_j} $$

In [ ]:
corr_matrix = returns_df.corr()
print(corr_matrix)

### Let us plot the correlation matrix ###

In [ ]:
create_correlation_heatmap(corr_matrix)

### Set up an optimal portfolio simulation calculation ###

In [ ]:
annual_returns = returns_df.mean() * 252
annual_cov_matrix = returns_df.cov() * 252
risk_free_rate = 0.03
sim_runs = 1000000
n = len(returns_df.columns)
print(n)

# I will set up placeholders to store all outputs of the simulation calculation:
weights_runs = np.zeros((sim_runs, n))
sharpe_ratio_runs = np.zeros(sim_runs)
expected_portfolio_returns_runs = np.zeros(sim_runs)
volatility_runs = np.zeros(sim_runs)

## Run the calculations to obtain an efficient frontier and the optimal portfolio

The Monte Carlo simulation approach used to generate the efficient frontier in this notebook was inspired by the *CFA Institute Python Programming Fundamentals* Practical Skills Module taught by Dr. Ryan Ahmed.

I have also implemented a similar Monte Carlo efficient frontier workflow in a separate project: my Django Advanced final project, **Equity Optimizer App** (GitHub: https://github.com/Kamend1/equity-optimizer-app).

A key benefit of the *Math for Developers* course is that it improved my understanding of vectors, matrices, and quadratic forms. After revisiting my earlier implementation, I realized that the simulation engine performed redundant matrix/vector operations. Refactoring the code to remove those unnecessary operations significantly improved performance. In the earlier version, approximately 30,000 simulations required ~20–30 minutes. After the refactor, the engine can run 1,000,000 simulations in under ~20 minutes on the same machine. This matters because a denser Monte Carlo cloud produces a smoother and more informative approximation of the feasible set and the efficient frontier.

In the setup section above, the parameter `sim_runs` controls the number of random portfolios generated. For development and quick validation, `100_000` runs are sufficient. For final charts and conclusions, `1_000_000` runs produce the cleanest frontier approximation.

In [2]:
for i in range(sim_runs):
    # Generate random weights
    weights = p_sim.generate_portfolio_weights(n)
    # Store the weights
    weights_runs[i, :] = weights

    # Call "simulation_engine" function and store Sharpe ratio, return, and volatility
    expected_portfolio_returns_runs[i], volatility_runs[i], sharpe_ratio_runs[i], \
     = p_sim.simulation_engine(weights, annual_returns, annual_cov_matrix, risk_free_rate)

    if i % 250 == 0:
        print(f"Simulation Run = {i}")
        print(f"Weights = {weights_runs[i].round(3)},\n Sharpe Ratio = {sharpe_ratio_runs[i]:.5f},\n"
             f"Expected return = {expected_portfolio_returns_runs[i]},\n"
             f"Volatility = {volatility_runs[i]}")
        print('\n')

NameError: name 'sim_runs' is not defined

### In this step, we extract the output data produced in the simulation calculation ###

- There is a problem - what if Sharpe Ratio is a negative?! #TODO

In [ ]:
sim_out_df = pd.DataFrame({'Volatility': volatility_runs.tolist(), 
                           'Portfolio_Return': expected_portfolio_returns_runs.tolist(), 
                           'Sharpe_Ratio': sharpe_ratio_runs.tolist() })
print(sim_out_df)

max_sharpe_idx = sim_out_df["Sharpe_Ratio"].idxmax()
optimal_portfolio = sim_out_df.loc[max_sharpe_idx]
cloud_df = sim_out_df.drop(index=max_sharpe_idx)
optimal_portfolio_weights = weights_runs[max_sharpe_idx]

print(optimal_portfolio)

### We will now plot the results and make some interesting observations ###

In [ ]:
fig = px.scatter(
    cloud_df,
    x="Volatility",
    y="Portfolio_Return",
    color="Sharpe_Ratio",
    size="Sharpe_Ratio",
    opacity=0.5,
    hover_data=["Sharpe_Ratio"]
)

fig.add_trace(go.Scatter(
    x=[optimal_portfolio["Volatility"]],
    y=[optimal_portfolio["Portfolio_Return"]],
    mode="markers+text",
    marker=dict(color="red", size=25, line=dict(color="orange", width=2)),
    name="Optimal Portfolio",
    text=["Optimal"],
    textposition="top left"
))

fig.update_layout(plot_bgcolor="white")
fig.show()

# Plot interactive plot for volatility
fig = px.line(sim_out_df, y = 'Volatility')
fig.show()

# Plot interactive plot for Portfolio Return
fig = px.line(sim_out_df, y = 'Portfolio_Return')
fig.update_traces(line_color = 'red')
fig.show()

# Plot interactive plot for Portfolio Return
fig = px.line(sim_out_df, y = 'Sharpe_Ratio')
fig.update_traces(line_color = 'purple')
fig.show()

### Finally, let's print and take not of the optimal portfolio weights ###

In [ ]:
print(list(zip(tickers, optimal_portfolio_weights)))

### Conclusion ###

So far we have used very little math. Most of us in the finance profession are taught to understand intuitively the concept and find practical solutions, without fully understanding the linear algebra and the geometry behind it. I substituted the actual solution produced by Markowitz in 1952, where he optimized to find the portfolio weights of a portfolio including predefined risk-bearing assets by finding the lowest-variance portfolio for each targeted level for portfolio return.

Instead of solving, I just produced n different portfolios by randomly generating weights, calculted their variance and returns and then plotted each return-variance combination on a scatter plot. The bright yellow dots appear to form a parabolic shape, which actually defines the efficient frontier defined by Markowitz. The greater n is in term of simulation runs, the closer the result is to the actual outcome. I will leave the variable sim_runs at 10 000, but you can try even higher. For example I tried 30 000 runs, it took around 20 minutes to calculate. But this is brute force, not mathematics. 

Let's now move to notebook 1_2 in order to observe how the theory and its supporting mathematics actually work. 